# Extended Exploratory Data Analysis (EDA)

This notebook visualizes the **synthetically engineered columns** that were added to the dataset.  We explore each column on its own, compare them against each other, and relate them to the original metrics.

In [ ]:
import pandas as pd, numpy as np
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

# Load the enriched dataset
df = pd.read_csv(r'../data/human_decision_fatigue_dataset_enriched.csv')

# List of new engineered columns
new_cols = [
    'Years_at_Company',
    'Self_Reported_Sleep_Quality',
    'Break_Room_Entry_Count',
    'Water_Dispenser_Refills',
    'Vending_Machine_Sugar_Purchases',
    'Corporate_Gym_Entry_Mins',
    'Peer_Collaboration_Pings',
    'Mid_Shift_Mood_Score'
]

## 1️⃣ Individual visualisations of the new columns

In [ ]:
for col in new_cols:
    if pd.api.types.is_numeric_dtype(df[col]):
        fig = px.histogram(df, x=col, nbins=30, title=f'Distribution of {col}')
        fig.update_layout(bargap=0.1)
        fig.show()
    else:
        fig = px.bar(df[col].value_counts().reset_index(),
                     x='index', y=col, title=f'Counts of {col}')
        fig.update_xaxes(title=col)
        fig.update_yaxes(title='Count')
        fig.show()

## 2️⃣ Pairwise relationships among the engineered **numeric** columns

In [ ]:
numeric_new = [c for c in new_cols if pd.api.types.is_numeric_dtype(df[c])]
fig = px.scatter_matrix(df, dimensions=numeric_new,
               title='Scatter matrix of engineered numeric columns')
fig.update_traces(diagonal_visible=False)
fig.show()

## 3️⃣ Relating engineered columns to original metrics
We will plot each numeric engineered column against a few key original variables: `Error_Rate`, `Cognitive_Load_Score`, `Decision_Fatigue_Score`, `Avg_Decision_Time_sec`, `Stress_Level_1_10`, `Hours_Awake`, `Task_Switches`, `Decisions_Made`.

In [ ]:
old_numeric = [
    'Error_Rate',
    'Cognitive_Load_Score',
    'Decision_Fatigue_Score',
    'Avg_Decision_Time_sec',
    'Stress_Level_1_10',
    'Hours_Awake',
    'Task_Switches',
    'Decisions_Made'
]
for new in numeric_new:
    for old in old_numeric:
        fig = px.scatter(df, x=new, y=old,
                     trendline='ols',
                     opacity=0.7,
                     title=f'{new} vs {old}')
        fig.update_layout(width=500, height=400)
        fig.show()

## 4️⃣ Highlighted interaction examples (optional)

In [ ]:
# Interaction 1 – Water + Caffeine → Error Rate
mask = (df['Caffeine_Intake_Cups'] > 2) & (df['Water_Dispenser_Refills'] < 3)
fig = px.box(df[mask], y='Error_Rate',
             title='Error Rate when Caffeine>2 cups & Water<3 refills')
fig.show()

# Interaction 2 – Break Room entries vs Decision Fatigue (quartiles)
break_bins = pd.qcut(df['Break_Room_Entry_Count'], q=4, duplicates='drop')
fig = px.box(df, x=break_bins, y='Decision_Fatigue_Score',
             title='Decision Fatigue across Break‑Room entry quartiles')
fig.update_xaxes(title='Break‑Room Entry Quartile')
fig.show()